# MSDS670 Final Project: Marvel by the Numbers
## Data Visualization Final Project

**Student:** Jose Lugo Gonzalez  
**Project Focus:** Marvel Cinematic Universe box office performance  
**Main Question:** How has MCU movie performance changed over time based on revenue, budget, and release era?

## Project Framing

I chose the Marvel Cinematic Universe because it is a fun topic, but it also works well as a business data visualization project. The MCU is one of the largest entertainment franchises in modern film, and its performance can be studied through revenue, production budget, release timing, and movie-level results.

For this project, I wanted to look at Marvel the way a data analyst would. Instead of only asking which movie was my favorite, I focused on how the franchise performed financially over time, which movies drove the biggest results, and whether bigger production budgets always led to bigger worldwide box office outcomes.

## Visualizations Required

1. Line plot
2. Bar plot
3. Scatterplot

All visualizations are built directly with Matplotlib using the object-oriented interface with `fig, ax = plt.subplots()`. The charts avoid pie charts, circle charts, 3D styling, and gridlines.


In [ ]:
# Import libraries
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator

---

## Part 1: Load the Data

For this project, I used a Marvel Cinematic Universe box office dataset built from The Numbers MCU franchise table. The dataset gives one row per MCU movie and includes release date, production budget, opening weekend, domestic box office, and worldwide box office.

This works well for the final project because it supports time-series analysis, sorted categorical comparisons, and a numeric scatterplot relationship.


In [ ]:
# Load dataset
mcu = pd.read_csv("data/mcu_box_office.csv")

# Preview the data
mcu.head()

In [ ]:
# Check dataset shape
print("MCU dataset shape:", mcu.shape)

In [ ]:
# Review columns
print(mcu.columns)

---

## Part 2: Prepare the Data

The original data stores dollar values as full numbers. For cleaner charts, I created dollar values in millions and billions. I also created a release year column and an analysis-defined release era column.

The release eras are not official Marvel phases. They are simple analysis groups used to compare the franchise across time:

1. Foundation: 2008 to 2012  
2. Expansion: 2013 to 2017  
3. Peak: 2018 to 2019  
4. Post-Endgame: 2021 to 2025  


In [ ]:
# Convert release date to datetime
mcu['release_date'] = pd.to_datetime(mcu['release_date'])
mcu['release_year'] = mcu['release_date'].dt.year

# Create dollar columns in millions for easier chart formatting
money_columns = [
    'production_budget',
    'opening_weekend_us',
    'domestic_box_office',
    'worldwide_box_office'
]

for column in money_columns:
    mcu[column + '_m'] = mcu[column] / 1_000_000

# Create a dollar column in billions for easier labels
mcu['worldwide_box_office_b'] = mcu['worldwide_box_office'] / 1_000_000_000

# Create return multiple
mcu['return_multiple'] = mcu['worldwide_box_office'] / mcu['production_budget']

# Sort movies by release date
mcu = mcu.sort_values('release_date').reset_index(drop=True)

mcu[['movie', 'release_year', 'production_budget_m', 'worldwide_box_office_m', 'return_multiple']].head()

In [ ]:
# Create analysis-defined MCU release eras
era_bins = [2007, 2012, 2017, 2019, 2025]
era_labels = [
    'Foundation\n2008-2012',
    'Expansion\n2013-2017',
    'Peak\n2018-2019',
    'Post-Endgame\n2021-2025'
]

mcu['release_era'] = pd.cut(
    mcu['release_year'],
    bins=era_bins,
    labels=era_labels,
    include_lowest=True
)

mcu[['movie', 'release_year', 'release_era']].tail()

In [ ]:
# Create annual revenue summary for the line plot
annual_revenue = (
    mcu
    .groupby('release_year', as_index=False)
    .agg(
        movie_count=('movie', 'count'),
        total_worldwide_revenue_m=('worldwide_box_office_m', 'sum'),
        avg_worldwide_revenue_m=('worldwide_box_office_m', 'mean')
    )
    .sort_values('release_year')
)

annual_revenue.head()

In [ ]:
# Create top 10 movies summary for the horizontal bar plot
top_10_movies = (
    mcu
    .sort_values('worldwide_box_office_m', ascending=False)
    .head(10)
    .sort_values('worldwide_box_office_m', ascending=True)
    .copy()
)

top_10_movies[['movie', 'worldwide_box_office_m']].head()

In [ ]:
# Create release era summary for the optional fourth visualization
era_summary = (
    mcu
    .groupby('release_era', observed=True)
    .agg(
        movie_count=('movie', 'count'),
        avg_worldwide_revenue_m=('worldwide_box_office_m', 'mean'),
        total_worldwide_revenue_m=('worldwide_box_office_m', 'sum')
    )
    .reset_index()
    .sort_values('avg_worldwide_revenue_m', ascending=True)
)

era_summary

In [ ]:
# Save cleaned analysis datasets for reference
Path("data").mkdir(exist_ok=True)

mcu.to_csv("data/mcu_box_office_cleaned.csv", index=False)
annual_revenue.to_csv("data/mcu_annual_revenue_summary.csv", index=False)
era_summary.to_csv("data/mcu_era_summary.csv", index=False)

---

## Part 3: Set Up the Matplotlib Theme

I wanted the project to feel connected to Marvel without making the charts too busy. The color palette uses deep red, gold, charcoal, and light gray. This gives the visuals a Marvel-inspired style while still keeping the charts professional.

The chart style also follows the final project rubric by removing gridlines and avoiding 3D styling.


In [ ]:
# Marvel inspired color palette
MARVEL_RED = '#B11313'
DARK_RED = '#7A0C0C'
GOLD = '#F5C542'
CHARCOAL = '#202020'
LIGHT_GRAY = '#F4F4F4'
MID_GRAY = '#A6A6A6'
WHITE = '#FFFFFF'

# Basic Matplotlib settings
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 16,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white'
})

# Create image folder for exported charts
Path("images").mkdir(exist_ok=True)

# Helper functions for clean formatting
def dollars_in_billions(x, pos=None):
    return f"${x / 1000:.1f}B"

def dollars_in_millions(x, pos=None):
    return f"${x:,.0f}M"

def clean_movie_title(title, max_length=34):
    if len(title) <= max_length:
        return title
    return title[:max_length - 3] + "..."

def clean_axis(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(MID_GRAY)
    ax.spines['bottom'].set_color(MID_GRAY)
    ax.tick_params(colors=CHARCOAL)
    ax.yaxis.label.set_color(CHARCOAL)
    ax.xaxis.label.set_color(CHARCOAL)
    ax.title.set_color(CHARCOAL)
    ax.grid(False)

def add_source_note(fig):
    fig.text(
        0.01,
        0.01,
        "Source: The Numbers MCU franchise box office table. Dollar values are nominal and not inflation adjusted.",
        fontsize=8,
        color='dimgray'
    )

---

## Visualization 1: Line Plot

### Question

How has annual worldwide MCU box office revenue changed over time?


In [ ]:
# Visualization 1: Annual worldwide MCU box office revenue over time
fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

ax.plot(
    annual_revenue['release_year'],
    annual_revenue['total_worldwide_revenue_m'],
    marker='o',
    linewidth=2.5,
    color=MARVEL_RED,
    markerfacecolor=GOLD,
    markeredgecolor=DARK_RED
)

# Highlight peak revenue year
peak_year = annual_revenue.loc[annual_revenue['total_worldwide_revenue_m'].idxmax()]

ax.scatter(
    [peak_year['release_year']],
    [peak_year['total_worldwide_revenue_m']],
    s=160,
    color=GOLD,
    edgecolor=DARK_RED,
    zorder=5
)

ax.annotate(
    f"Peak year\n{int(peak_year['release_year'])}: ${peak_year['total_worldwide_revenue_m'] / 1000:.1f}B",
    xy=(peak_year['release_year'], peak_year['total_worldwide_revenue_m']),
    xytext=(peak_year['release_year'] - 5.5, peak_year['total_worldwide_revenue_m'] - 900),
    arrowprops=dict(arrowstyle='->', color=DARK_RED, lw=1.5),
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.35', fc=LIGHT_GRAY, ec=GOLD, alpha=0.95)
)

# Add rounded data labels
for _, row in annual_revenue.iterrows():
    ax.text(
        row['release_year'],
        row['total_worldwide_revenue_m'] + 110,
        f"${row['total_worldwide_revenue_m'] / 1000:.1f}B",
        ha='center',
        va='bottom',
        fontsize=8,
        color=CHARCOAL
    )

ax.set_title("Annual Worldwide MCU Box Office Revenue by Release Year")
ax.set_xlabel("Release Year")
ax.set_ylabel("Worldwide Box Office Revenue (USD Billions)")
ax.yaxis.set_major_formatter(FuncFormatter(dollars_in_billions))
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

clean_axis(ax)
add_source_note(fig)

fig.savefig("images/01_annual_worldwide_revenue_line.png", dpi=300, bbox_inches='tight')
plt.show()

### Interpretation

The line plot shows that MCU revenue did not stay consistent every year. Annual worldwide box office revenue grew as the franchise became more connected, reached its strongest point during the Avengers-era peak, and then became more uneven after 2019. This chart gives the project its main story because it shows growth, peak performance, and a changing post-Endgame period.


---

## Visualization 2: Horizontal Bar Plot

### Question

Which MCU movies generated the highest worldwide box office revenue?


In [ ]:
# Visualization 2: Top 10 MCU movies by worldwide box office revenue
top_10_movies['movie_short'] = top_10_movies['movie'].apply(clean_movie_title)

fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

bar_colors = [
    GOLD if movie == 'Avengers: Endgame' else MARVEL_RED
    for movie in top_10_movies['movie']
]

bars = ax.barh(
    top_10_movies['movie_short'],
    top_10_movies['worldwide_box_office_m'],
    color=bar_colors,
    edgecolor=DARK_RED,
    linewidth=0.6
)

# Add rounded data labels
for bar, value in zip(bars, top_10_movies['worldwide_box_office_m']):
    ax.text(
        value + 40,
        bar.get_y() + bar.get_height() / 2,
        f"${value / 1000:.1f}B",
        va='center',
        fontsize=10,
        fontweight='bold',
        color=CHARCOAL
    )

ax.set_title("Top 10 MCU Movies by Worldwide Box Office Revenue")
ax.set_xlabel("Worldwide Box Office Revenue (USD Billions)")
ax.set_ylabel("Movie")
ax.xaxis.set_major_formatter(FuncFormatter(dollars_in_billions))

clean_axis(ax)
add_source_note(fig)

fig.savefig("images/02_top_10_worldwide_movies_bar.png", dpi=300, bbox_inches='tight')
plt.show()

### Interpretation

The sorted horizontal bar chart shows that the largest MCU box office results are concentrated in a small group of films. Avengers: Endgame is the clear leader, and many of the top movies are major crossover films. This supports the idea that the MCU's biggest financial outcomes were driven by event-style releases rather than evenly spread across every movie.


---

## Visualization 3: Scatterplot

### Question

Do higher production budgets usually lead to higher worldwide box office revenue?


In [ ]:
# Visualization 3: Production budget vs worldwide box office revenue
fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

ax.scatter(
    mcu['production_budget_m'],
    mcu['worldwide_box_office_m'],
    s=85,
    alpha=0.78,
    color=MARVEL_RED,
    edgecolor=GOLD,
    linewidth=1.1
)

# Add a simple trend line
x = mcu['production_budget_m']
y = mcu['worldwide_box_office_m']
z = np.polyfit(x, y, 1)
p = np.poly1d(z)

x_line = np.linspace(x.min(), x.max(), 100)

ax.plot(
    x_line,
    p(x_line),
    color=GOLD,
    linewidth=2.2,
    alpha=0.95
)

# Annotate key movies
label_movies = [
    'Avengers: Endgame',
    'Avengers: Infinity War',
    'Spider-Man: No Way Home',
    'The Marvels'
]

for movie in label_movies:
    row = mcu[mcu['movie'] == movie].iloc[0]

    ax.annotate(
        clean_movie_title(row['movie'], max_length=24),
        xy=(row['production_budget_m'], row['worldwide_box_office_m']),
        xytext=(row['production_budget_m'] + 8, row['worldwide_box_office_m'] + 80),
        arrowprops=dict(arrowstyle='->', color=DARK_RED, lw=1.2),
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', fc=LIGHT_GRAY, ec=GOLD, alpha=0.95)
    )

ax.set_title("Production Budget vs. Worldwide MCU Box Office Revenue")
ax.set_xlabel("Production Budget (USD Millions)")
ax.set_ylabel("Worldwide Box Office Revenue (USD Millions)")
ax.xaxis.set_major_formatter(FuncFormatter(dollars_in_millions))
ax.yaxis.set_major_formatter(FuncFormatter(dollars_in_millions))

clean_axis(ax)
add_source_note(fig)

fig.savefig("images/03_budget_vs_worldwide_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

### Interpretation

The scatterplot shows a general positive relationship between production budget and worldwide revenue, but the relationship is not perfect. Some movies earned far more than their budgets would suggest, while other high-budget movies did not reach the top of the revenue range. This adds business nuance because it shows that spending more money does not automatically guarantee a stronger box office result.


---

## Visualization 4: Release Era Bar Plot

### Question

Which MCU release era had the strongest average worldwide box office performance?


In [ ]:
# Visualization 4: Average worldwide revenue by analysis-defined release era
fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

era_colors = [
    GOLD if 'Peak' in str(era) else MARVEL_RED
    for era in era_summary['release_era']
]

bars = ax.barh(
    era_summary['release_era'],
    era_summary['avg_worldwide_revenue_m'],
    color=era_colors,
    edgecolor=DARK_RED,
    linewidth=0.6
)

# Add rounded data labels with movie counts
for bar, value, count in zip(
    bars,
    era_summary['avg_worldwide_revenue_m'],
    era_summary['movie_count']
):
    ax.text(
        value + 30,
        bar.get_y() + bar.get_height() / 2,
        f"${value / 1000:.1f}B avg | n={count}",
        va='center',
        fontsize=10,
        fontweight='bold',
        color=CHARCOAL
    )

ax.set_title("Average Worldwide MCU Box Office Revenue by Release Era")
ax.set_xlabel("Average Worldwide Box Office Revenue (USD Billions)")
ax.set_ylabel("Release Era")
ax.xaxis.set_major_formatter(FuncFormatter(dollars_in_billions))

clean_axis(ax)
add_source_note(fig)

fig.savefig("images/04_average_revenue_by_era_bar.png", dpi=300, bbox_inches='tight')
plt.show()

### Interpretation

The release-era bar plot shows that the 2018 to 2019 peak period had the strongest average worldwide box office performance. This makes sense because this era includes major event films that had unusually high audience demand. The post-Endgame era still includes successful movies, but the average performance is lower and more mixed than the peak period.


---

## Final Conclusions

1. The MCU grew into a major box office franchise over time, with annual worldwide revenue reaching its highest point during the 2018 to 2019 Avengers-era peak.

2. The top box office results were not evenly spread across all Marvel movies. A small number of major event films created the largest revenue outcomes.

3. Production budget is related to worldwide revenue, but it does not fully explain success. Audience demand, timing, franchise momentum, and movie importance also appear to matter.

## Overall Story

As a Marvel fan, it is easy to think about the movies through characters, storylines, and favorite scenes. Looking at the data made me see the MCU differently. The franchise was also a business system that built momentum over time. The data shows that Marvel's strongest years came when the franchise turned individual movies into major events, but the post-Endgame period has been more uneven.
